In [0]:
dbutils.widgets.text("catalog", "fiap_analytics")
dbutils.widgets.text("schema_ml", "ml")

CATALOG   = dbutils.widgets.get("catalog")
SCHEMA_ML = dbutils.widgets.get("schema_ml")
S  = f"{CATALOG}.silver.incidentes"
ML = f"{CATALOG}.{SCHEMA_ML}"

import numpy as np
import pandas as pd

print(f"Schema ML: {ML}")

Schema ML: fiap_analytics.ml


In [0]:
# Lê as previsões já gravadas e junta a prioridade (que a tabela de regressão não tem).
dados = spark.sql(f"""
SELECT r.numero,
       r.duracao_real_seg,
       r.duracao_prevista_seg,
       s.prioridade_cod
FROM {ML}.regressao_linear r
JOIN {S} s ON s.numero = r.numero
WHERE r.modelo = 'regressao_linear'
""").toPandas()

print(f"Chamados avaliados: {len(dados):,}")
display(dados.head())

Chamados avaliados: 1,225


numero,duracao_real_seg,duracao_prevista_seg,prioridade_cod
INC7830303,5.2313054000000015E7,148920.18274340348,3
INC8154800,3.207063999999998E7,519997.93509468285,3
INC8301589,2.084422199999999E7,517920.25459900906,3
INC8434371,1.2266755999999996E7,4752173.362797699,3
INC8442105,1.4938376999999993E7,6358862.573147797,3


In [0]:
# Prazos do KPI por prioridade, do dicionário de dados (em segundos).
# 1 Crítica=4h | 2 Alta=4h | 3 Média=12h | 4 Baixa=24h | 5 Muito Baixa=96h
PRAZO_SEG = {1: 4*3600, 2: 4*3600, 3: 12*3600, 4: 24*3600, 5: 96*3600}

dados['prazo_seg']   = dados['prioridade_cod'].map(PRAZO_SEG)
dados['razao_prazo'] = (dados['duracao_prevista_seg'] / dados['prazo_seg']).round(2)

# Risco previsto na abertura: a duração estimada passa do prazo?
dados['risco_violacao'] = (dados['duracao_prevista_seg'] > dados['prazo_seg']).astype(int)

def faixa(r):
    if r >= 1.0:  return '1_ALTO (prevê estouro)'
    if r >= 0.7:  return '2_ATENCAO (>70% do prazo)'
    if r >= 0.4:  return '3_MODERADO'
    return '4_BAIXO'

dados['faixa_risco'] = dados['razao_prazo'].apply(faixa)

display(dados.groupby('faixa_risco').agg(
    chamados=('numero','size'),
    dur_prev_media_h=('duracao_prevista_seg', lambda s: round(s.mean()/3600,1))
).reset_index())

faixa_risco,chamados,dur_prev_media_h
1_ALTO (prevê estouro),257,783.5
2_ATENCAO (>70% do prazo),49,9.2
3_MODERADO,92,5.4
4_BAIXO,827,1.8


In [0]:
# Compara o risco previsto na abertura com o que de fato aconteceu (duração real vs prazo).
# Usa violação por tempo CORRIDO, para ser coerente com a duração corrida que a regressão prevê.
dados['violou_real'] = (dados['duracao_real_seg'] > dados['prazo_seg']).astype(int)

from sklearn.metrics import confusion_matrix, classification_report

print("Violações reais no conjunto (tempo corrido):",
      dados['violou_real'].value_counts().to_dict(), "\n")

cm = confusion_matrix(dados['violou_real'], dados['risco_violacao'])
print(pd.DataFrame(cm,
      index=['real: não violou','real: violou'],
      columns=['prev: não','prev: sim']), "\n")

print(classification_report(dados['violou_real'], dados['risco_violacao'],
      target_names=['não violou','violou'], zero_division=0))

Violações reais no conjunto (tempo corrido): {0: 929, 1: 296} 

                  prev: não  prev: sim
real: não violou        882         47
real: violou             86        210 

              precision    recall  f1-score   support

  não violou       0.91      0.95      0.93       929
      violou       0.82      0.71      0.76       296

    accuracy                           0.89      1225
   macro avg       0.86      0.83      0.84      1225
weighted avg       0.89      0.89      0.89      1225



In [0]:
import mlflow
from sklearn.metrics import precision_score, recall_score, f1_score

mlflow.set_experiment(f"/Users/{spark.sql('SELECT current_user()').first()[0]}/fiap_risco_sla")

with mlflow.start_run(run_name="risco_violacao_sla"):
    mlflow.log_param("abordagem", "duracao_prevista_vs_prazo_dicionario")
    mlflow.log_param("fonte_duracao", "regressao_linear")
    mlflow.log_param("prazos_seg", str(PRAZO_SEG))

    y_true = dados['violou_real']
    y_pred = dados['risco_violacao']
    mlflow.log_metric("precision_violacao", precision_score(y_true, y_pred, zero_division=0))
    mlflow.log_metric("recall_violacao",    recall_score(y_true, y_pred, zero_division=0))
    mlflow.log_metric("f1_violacao",        f1_score(y_true, y_pred, zero_division=0))
    mlflow.log_metric("n_avaliados", len(dados))

    faixas = (dados.groupby('faixa_risco')
              .agg(chamados=('numero', 'size'),
                   violaram=('violou_real', 'sum'))
              .assign(pct_violacao=lambda d: (100*d.violaram/d.chamados).round(1))
              .reset_index())
    faixas.to_csv("/tmp/faixas_risco.csv", index=False)
    mlflow.log_artifact("/tmp/faixas_risco.csv")

    print("Risco de SLA registrado no MLflow")
    print(faixas)

Risco de SLA registrado no MLflow
                 faixa_risco  chamados  violaram  pct_violacao
0     1_ALTO (prevê estouro)       257       210          81.7
1  2_ATENCAO (>70% do prazo)        49        13          26.5
2                 3_MODERADO        92        15          16.3
3                    4_BAIXO       827        58           7.0


In [0]:
saida = dados[['numero','prioridade_cod','prazo_seg',
               'duracao_prevista_seg','duracao_real_seg',
               'razao_prazo','faixa_risco','risco_violacao','violou_real']].copy()
saida['dt_geracao'] = pd.Timestamp.now()

(spark.createDataFrame(saida)
      .write.format("delta").mode("overwrite").option("overwriteSchema","true")
      .saveAsTable(f"{ML}.risco_violacao"))

print(f"{len(saida):,} chamados gravados em {ML}.risco_violacao")

# Quanto o risco previsto bate com a violação real, por faixa
display(spark.sql(f"""
SELECT faixa_risco,
       count(*)                                       AS chamados,
       sum(violou_real)                               AS violaram_de_fato,
       round(100.0*sum(violou_real)/count(*), 1)      AS pct_violacao_real
FROM {ML}.risco_violacao
GROUP BY faixa_risco ORDER BY faixa_risco
"""))

1,225 chamados gravados em fiap_analytics.ml.risco_violacao


faixa_risco,chamados,violaram_de_fato,pct_violacao_real
1_ALTO (prevê estouro),257,210,81.7
2_ATENCAO (>70% do prazo),49,13,26.5
3_MODERADO,92,15,16.3
4_BAIXO,827,58,7.0
